In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier
import pickle
import os


In [2]:
import os
BASE_DIR = "/workspaces/Aviation-Dashboard/backend"
DATA_PATH = os.path.join(BASE_DIR, "data", "data","morocco_flights.csv")
MODEL_DIR = os.path.join(BASE_DIR, "app", "ml")


In [3]:
df = pd.read_csv(DATA_PATH)
print(df.head())
print(df.info())
print(df.describe())
print(df.isnull().sum())



  origin destination airline  duration_min  month  day_of_week  dep_hour  \
0    CMN         CAI     RAM           223     10            6         5   
1    CMN         AGA     RAM            53      3            4        10   
2    AGA         ORY     RAM           167     12            2        17   
3    STN         RAK     RYR           224     11            6         9   
4    CMN         MAD     RAM           118     12            4        16   

   delayed  
0        0  
1        0  
2        1  
3        0  
4        0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   origin        80000 non-null  object
 1   destination   80000 non-null  object
 2   airline       80000 non-null  object
 3   duration_min  80000 non-null  int64 
 4   month         80000 non-null  int64 
 5   day_of_week   80000 non-null  int64 
 6   dep_hour      80

In [4]:
df.dtypes

origin          object
destination     object
airline         object
duration_min     int64
month            int64
day_of_week      int64
dep_hour         int64
delayed          int64
dtype: object

In [5]:
encoders = {}
categorical_cols = ['origin', 'destination', 'airline']

for col in categorical_cols:
    le = LabelEncoder()
    df[f'{col}_encoded'] = le.fit_transform(df[col])
    encoders[col] = le
    print(f"   {col}: {len(le.classes_)} unique values")

# Prepare features and target
# REMOVED: 'distance_km' and 'is_international'
feature_cols = [
    'origin_encoded', 'destination_encoded', 'airline_encoded',
    'month', 'day_of_week', 'dep_hour'
]

X = df[feature_cols]
y = df['delayed'].astype(int)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 Training set: {len(X_train)} flights")
print(f"📊 Test set: {len(X_test)} flights")

   origin: 13 unique values
   destination: 31 unique values
   airline: 4 unique values

📊 Training set: 64000 flights
📊 Test set: 16000 flights


In [6]:
mlflow.set_experiment("flight_delay_prediction")

<Experiment: artifact_location='/workspaces/Aviation-Dashboard/backend/app/ml/mlruns/1', creation_time=1778075692330, experiment_id='1', last_update_time=1778075692330, lifecycle_stage='active', name='flight_delay_prediction', tags={}, trace_location=None, workspace='default'>

In [7]:
print(mlflow.get_tracking_uri())

sqlite:////workspaces/Aviation-Dashboard/backend/app/ml/mlflow.db


In [8]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [9]:
print(mlflow.get_tracking_uri())

http://127.0.0.1:5000


In [10]:
mlflow.set_experiment("flight_delay_prediction")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1778080941210, experiment_id='1', last_update_time=1778080941210, lifecycle_stage='active', name='flight_delay_prediction', tags={}, trace_location=None, workspace='default'>

In [11]:
from sklearn.metrics import recall_score, precision_score, f1_score

with mlflow.start_run(run_name="logistic_regression") as run:
    # Train logistic regression model
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    # Predict and evaluate
    y_pred = model.predict(X_test)
    
    #compute metrics : recall precision f1-score 
    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Log parameters and metrics to MLflow
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("f1_score", f1)
    

    # Log the model itself
    mlflow.sklearn.log_model(model, "logistic_regression_model")

2026/05/06 15:45:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 15:45:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/06 15:46:00 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run logistic_regression at: http://127.0.0.1:5000/#/experiments/1/runs/4347f0a7545343418a1220a88c093bac
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [12]:
with mlflow.start_run(run_name="random_forest") as run:
    rf= RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)
    accuracy_rf = accuracy_score(y_test, y_pred_rf)
    roc_auc_rf = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
    recall_rf = recall_score(y_test, y_pred_rf)
    precision_rf = precision_score(y_test, y_pred_rf)
    f1_rf = f1_score(y_test, y_pred_rf)
    mlflow.log_param("model_type", "Random Forest")
    mlflow.log_metric("accuracy", accuracy_rf)
    mlflow.log_metric("roc_auc", roc_auc_rf)
    mlflow.log_metric("recall", recall_rf)
    mlflow.log_metric("precision", precision_rf)
    mlflow.log_metric("f1_score", f1_rf)
    mlflow.sklearn.log_model(rf, "random_forest_model")
    

2026/05/06 15:46:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 15:46:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/06 15:46:16 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run random_forest at: http://127.0.0.1:5000/#/experiments/1/runs/a67b797b56b545d6be5d5ad24a4bff8e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [13]:
with mlflow.start_run(run_name="xgboost") as run:
    xgb_model = xgb.XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss')
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)
    accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
    roc_auc_xgb = roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1])
    recall_xgb = recall_score(y_test, y_pred_xgb)
    precision_xgb = precision_score(y_test, y_pred_xgb)
    f1_xgb = f1_score(y_test, y_pred_xgb)
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_metric("accuracy", accuracy_xgb)
    mlflow.log_metric("roc_auc", roc_auc_xgb)
    mlflow.log_metric("recall", recall_xgb)
    mlflow.log_metric("precision", precision_xgb)
    mlflow.log_metric("f1_score", f1_xgb)
    mlflow.sklearn.log_model(xgb_model, "xgboost_model")

/workspaces/Aviation-Dashboard/backend/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [15:46:22] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026/05/06 15:46:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 15:46:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/06 15:46:26 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run xgboost at: http://127.0.0.1:5000/#/experiments/1/runs/afd983df2ea241b5a5ce3955ca107b3d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [14]:
with mlflow.start_run(run_name="adaboost") as run:
    adabst = AdaBoostClassifier(n_estimators=100, random_state=42)
    adabst.fit(X_train, y_train)
    y_pred_adabst = adabst.predict(X_test)
    accuracy_adabst = accuracy_score(y_test, y_pred_adabst)
    roc_auc_adabst = roc_auc_score(y_test, adabst.predict_proba(X_test)[:, 1])
    recall_adabst = recall_score(y_test, y_pred_adabst)
    precision_adabst = precision_score(y_test, y_pred_adabst)
    f1_adabst = f1_score(y_test, y_pred_adabst)
    mlflow.log_param("model_type", "AdaBoost")
    mlflow.log_metric("accuracy", accuracy_adabst)
    mlflow.log_metric("roc_auc", roc_auc_adabst)
    mlflow.log_metric("recall", recall_adabst)
    mlflow.log_metric("precision", precision_adabst)
    mlflow.log_metric("f1_score", f1_adabst)
    mlflow.sklearn.log_model(adabst, "adaboost_model")

2026/05/06 15:46:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 15:46:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/06 15:46:31 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run adaboost at: http://127.0.0.1:5000/#/experiments/1/runs/f3138c7ee303442eb25608d8f6d185ab
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [16]:
with mlflow.start_run(run_name="SVM") as run:
    svm_model = SVC(probability=False, random_state=42)
    svm_model.fit(X_train, y_train)
    
    y_pred_svm = svm_model.predict(X_test)
    
    accuracy_svm = accuracy_score(y_test, y_pred_svm)
    recall_svm = recall_score(y_test, y_pred_svm)
    precision_svm = precision_score(y_test, y_pred_svm)
    f1_svm = f1_score(y_test, y_pred_svm)
    
    mlflow.log_param("model_type", "SVM")
    mlflow.log_metric("accuracy", accuracy_svm)
    mlflow.log_metric("recall", recall_svm)
    mlflow.log_metric("precision", precision_svm)
    mlflow.log_metric("f1_score", f1_svm)
    
    mlflow.sklearn.log_model(svm_model, "svm_model")

2026/05/06 15:51:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 15:51:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/06 15:51:51 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run SVM at: http://127.0.0.1:5000/#/experiments/1/runs/a13f8efcdacf424e81fc043c15242014
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [17]:
with mlflow.start_run(run_name="Knn") as run:
    knn_model = KNeighborsClassifier(n_neighbors=5)
    knn_model.fit(X_train, y_train)
    
    y_pred_knn = knn_model.predict(X_test)
    
    accuracy_knn = accuracy_score(y_test, y_pred_knn)
    recall_knn = recall_score(y_test, y_pred_knn)
    precision_knn = precision_score(y_test, y_pred_knn)
    f1_knn = f1_score(y_test, y_pred_knn)
    
    mlflow.log_param("model_type", "KNN")
    mlflow.log_metric("accuracy", accuracy_knn)
    mlflow.log_metric("recall", recall_knn)
    mlflow.log_metric("precision", precision_knn)
    mlflow.log_metric("f1_score", f1_knn)
    
    mlflow.sklearn.log_model(knn_model, "knn_model")

2026/05/06 15:53:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 15:53:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/06 15:53:36 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run Knn at: http://127.0.0.1:5000/#/experiments/1/runs/5b549000d4ce4b7aa3ef2350d9a0a57b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [18]:
with mlflow.start_run(run_name="GradientBoostingClassifier") as run:
    gbc_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
    gbc_model.fit(X_train, y_train)
    
    y_pred_gbc = gbc_model.predict(X_test)
    
    accuracy_gbc = accuracy_score(y_test, y_pred_gbc)
    roc_auc_gbc = roc_auc_score(y_test, gbc_model.predict_proba(X_test)[:, 1])
    recall_gbc = recall_score(y_test, y_pred_gbc)
    precision_gbc = precision_score(y_test, y_pred_gbc)
    f1_gbc = f1_score(y_test, y_pred_gbc)
    
    mlflow.log_param("model_type", "Gradient Boosting Classifier")
    mlflow.log_metric("accuracy", accuracy_gbc)
    mlflow.log_metric("roc_auc", roc_auc_gbc)
    mlflow.log_metric("recall", recall_gbc)
    mlflow.log_metric("precision", precision_gbc)
    mlflow.log_metric("f1_score", f1_gbc)
    
    mlflow.sklearn.log_model(gbc_model, "gradient_boosting_classifier_model")

2026/05/06 15:54:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 15:54:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/06 15:54:38 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run GradientBoostingClassifier at: http://127.0.0.1:5000/#/experiments/1/runs/d07c65daf29e4c1fa588c8d14eb15b82
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [19]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

In [20]:
with mlflow.start_run(run_name="GaussianNB") as run:
    nb_model = GaussianNB()
    nb_model.fit(X_train, y_train)
    
    y_pred_nb = nb_model.predict(X_test)
    
    accuracy_nb = accuracy_score(y_test, y_pred_nb)
    recall_nb = recall_score(y_test, y_pred_nb)
    precision_nb = precision_score(y_test, y_pred_nb)
    f1_nb = f1_score(y_test, y_pred_nb)
    
    mlflow.log_param("model_type", "Gaussian Naive Bayes")
    mlflow.log_metric("accuracy", accuracy_nb)
    mlflow.log_metric("recall", recall_nb)
    mlflow.log_metric("precision", precision_nb)
    mlflow.log_metric("f1_score", f1_nb)
    
    mlflow.sklearn.log_model(nb_model, "gaussian_nb_model")

2026/05/06 15:57:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 15:57:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/06 15:57:21 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run GaussianNB at: http://127.0.0.1:5000/#/experiments/1/runs/e81ecabbff6f4983b03856471ffaa518
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [21]:
with mlflow.start_run(run_name="DecisionTreeClassifier ") as run:
    dt_model = DecisionTreeClassifier(random_state=42)
    dt_model.fit(X_train, y_train)
    
    y_pred_dt = dt_model.predict(X_test)
    
    accuracy_dt = accuracy_score(y_test, y_pred_dt)
    recall_dt = recall_score(y_test, y_pred_dt)
    precision_dt = precision_score(y_test, y_pred_dt)
    f1_dt = f1_score(y_test, y_pred_dt)
    
    mlflow.log_param("model_type", "Decision Tree Classifier")
    mlflow.log_metric("accuracy", accuracy_dt)
    mlflow.log_metric("recall", recall_dt)
    mlflow.log_metric("precision", precision_dt)
    mlflow.log_metric("f1_score", f1_dt)
    
    mlflow.sklearn.log_model(dt_model, "decision_tree_classifier_model")

2026/05/06 15:58:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 15:58:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/06 15:58:21 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run DecisionTreeClassifier  at: http://127.0.0.1:5000/#/experiments/1/runs/ff7c4d0a9e88422e8a482e658d7d72bf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [ ]:
with mlflow.start_run(run_name="random_forest") as run:
    mlflow.set_tag("threshold", "0.4")
    #set threshold for random forest model
    y_pred_rf_proba = rf.predict_proba(X_test)[:, 1]
    y_pred_rf_threshold = (y_pred_rf_proba >= 0.4).astype(int)
    accuracy_rf_threshold = accuracy_score(y_test, y_pred_rf_threshold)
    recall_rf_threshold = recall_score(y_test, y_pred_rf_threshold)
    precision_rf_threshold = precision_score(y_test, y_pred_rf_threshold)
    f1_rf_threshold = f1_score(y_test, y_pred_rf_threshold)
    mlflow.log_metric("accuracy_threshold", accuracy_rf_threshold)
    mlflow.log_metric("recall_threshold", recall_rf_threshold)
    mlflow.log_metric("precision_threshold", precision_rf_threshold)
    mlflow.log_metric("f1_score_threshold", f1_rf_threshold)
    mlflow.sklearn.log_model(rf, "random_forest_model_threshold")

2026/05/06 16:01:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 16:01:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
